In [0]:
from pyspark.sql import *
from pyspark.sql import functions as F
from pyspark.sql.functions import *
from pyspark.sql.window import *
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

In [0]:
from datetime import date
from pyspark.sql.types import StructType, StructField, IntegerType, DateType
schema = StructType([

    StructField("order_id", IntegerType(), True),
    StructField("customer_id", IntegerType(), True),
    StructField("order_date", DateType(), True)
])

data = [
    (1, 101, date(2024, 1, 1)),
    (2, 101, date(2024, 1, 1)),
    (3, 102, date(2024, 1, 1)),
    (4, 102, date(2024, 1, 2)),
    (5, 103, date(2024, 1, 2)),
    (6, 103, date(2024, 1, 2))
]

df = spark.createDataFrame(
    data,
    schema
)
display(df)

##Answer
df.groupBy("customer_id", "order_date").agg(count("*").alias("order_count")).filter(col("order_count") > 1).show()

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

spark = SparkSession.builder.getOrCreate()

schema = StructType([
    StructField("emp_id", IntegerType(), True),
    StructField("emp_name", StringType(), True),
    StructField("department", StringType(), True),
    StructField("salary", IntegerType(), True)
])

data = [
    (1, "Amit", "IT", 60000),
    (2, "Neha", "IT", 75000),
    (3, "Rahul", "HR", 50000),
    (4, "Priya", "HR", 65000),
    (5, "Karan", "Finance", 80000)
]

df = spark.createDataFrame(data, schema)
df.show()

###Answer
window_spec = Window.partitionBy("department").orderBy(desc("salary"))
df1 = df.withColumn("row_num", dense_rank().over(window_spec)).filter(col("row_num") == 1)
df2 = df1.select("department", "emp_name", "salary")
display(df2)


In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType

spark = SparkSession.builder.getOrCreate()

schema = StructType([
    StructField("txn_id", IntegerType(), True),
    StructField("user_id", IntegerType(), True),
    StructField("amount", IntegerType(), True)
])

data = [
    (1, 101, 300),
    (2, 101, 400),
    (3, 101, 500),
    (4, 102, 200),
    (5, 102, 300),
    (6, 103, 1200)
]

df = spark.createDataFrame(data, schema)
df.show()

####Answer
df1 = df.groupBy("user_id").agg(sum("amount").alias("total_amount")).filter(col("total_amount") > 1000)
df1.show()


In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, DateType
from pyspark.sql.functions import col
from datetime import date

schema = StructType([
    StructField("user_id", IntegerType(), True),
    StructField("event_date", DateType(), True)
])

data = [
    (1, date(2024, 1, 1)),
    (1, date(2024, 1, 2)),
    (1, date(2024, 1, 4)),
    (2, date(2024, 1, 5)),
    (2, date(2024, 1, 6)),
    (3, date(2024, 1, 10))
]

df = spark.createDataFrame(
    data,
    schema
)

df_0 = df.withColumn("event_date", col("event_date").cast("date"))
display(df)

###Answer
window_spec = Window.partitionBy("user_id").orderBy(col("event_date"))
df_1 = df_0.withColumn("Previous_Day",lag("event_date").over(window_spec))
df_2 = df_1.filter(datediff(col("event_date"), col("Previous_Day")) == 1).select("user_id", "event_date")
df_2.show()

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, DateType, TimestampType
from pyspark.sql.functions import col
from datetime import date, datetime

schema = StructType([
    StructField("user_id", IntegerType(), True),
    StructField("login_date", DateType(), True),
    StructField("login_time", TimestampType(), True)
])

data = [
    (1, date(2024, 1, 1), datetime(2024, 1, 1, 9, 0, 0)),
    (1, date(2024, 1, 1), datetime(2024, 1, 1, 10, 30, 0)),
    (2, date(2024, 1, 2), datetime(2024, 1, 2, 8, 45, 0)),
    (2, date(2024, 1, 2), datetime(2024, 1, 2, 9, 15, 0))
]

df = spark.createDataFrame(
    data,
    schema
)

display(df)


###Answer
window_spec = Window.partitionBy("login_date","user_id").orderBy(desc("login_time"))
df1 = df.withColumn("row_num", dense_rank().over(window_spec)).filter(col("row_num") == 1)
df2 = df1.select("user_id", "login_date", "login_time")
df2.show()

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, DateType, DoubleType

schema = StructType([
    StructField("user_id", IntegerType(), True),
    StructField("txn_date", DateType(), True),
    StructField("amount", DoubleType(), True)
])

from datetime import date

data = [
    (1, date(2024, 1, 1), 100.0),
    (1, date(2024, 1, 3), 50.0),
    (1, date(2024, 1, 5), -20.0),
    (2, date(2024, 1, 2), 200.0),
    (2, date(2024, 1, 4), 100.0)
]

df = spark.createDataFrame(data, schema)

from pyspark.sql.window import Window
from pyspark.sql.functions import sum as _sum

window_spec = (
    Window.partitionBy("user_id")
          .orderBy("txn_date")
          .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

df_running = df.withColumn(
    "running_total",
    _sum("amount").over(window_spec)
).display()

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, DateType, DoubleType
from datetime import date

schema = StructType([
    StructField("order_id", IntegerType(), True),
    StructField("customer_id", IntegerType(), True),
    StructField("order_date", DateType(), True),
    StructField("amount", DoubleType(), True)
])

data = [
    (1, 101, date(2024, 1, 10), 200.0),
    (2, 101, date(2024, 2, 15), 150.0),
    (3, 102, date(2024, 1, 20), 300.0),
    (4, 103, date(2024, 3, 1), 400.0),
    (5, 102, date(2024, 2, 10), 100.0)
]

df = spark.createDataFrame(data, schema)

df_new = df.groupBy("customer_id").agg(sum("amount").alias("total_amount"), max("order_date").alias("latest_order_date")).orderBy("total_amount")
display(df_new)


In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType
from pyspark.sql.window import Window
from pyspark.sql.functions import dense_rank, desc, col

schema = StructType([
    StructField("emp_id", IntegerType(), True),
    StructField("emp_name", StringType(), True),
    StructField("dept_id", IntegerType(), True),
    StructField("salary", DoubleType(), True)
])
data = [
    (1, "A", 10, 10000),
    (2, "B", 10, 15000),
    (3, "C", 10, 12000),
    (4, "D", 20, 20000),
    (5, "E", 20, 18000),
    (6, "F", 20, 22000)
]

df = spark.createDataFrame(data, schema)

window_spec = Window.orderBy(desc("salary"))
df1 = df.withColumn("row_num", dense_rank().over(window_spec)).filter(col("row_num") == 2)
display(df1)


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import *
from datetime import date  # ✅ Fix 1: Missing import

schema = StructType([
    StructField("order_id", IntegerType(), True),      # ✅ Fix 2: Added missing fields
    StructField("customer_id", IntegerType(), True),   # ✅
    StructField("product_id", IntegerType(), True),
    StructField("order_date", DateType(), True),
    StructField("amount", DoubleType(), True)
])

data = [
    (1, 101, 10, date(2024, 1, 1), 200.0),
    (2, 101, 11, date(2024, 1, 5), 300.0),
    (3, 102, 10, date(2024, 1, 3), 150.0),
    (4, 101, 12, date(2024, 2, 1), 400.0),
    (5, 102, 11, date(2024, 2, 10), 250.0),
    (6, 103, 13, date(2024, 2, 15), 500.0),
    (7, 103, 14, date(2024, 3, 1), 100.0)
]

df = spark.createDataFrame(data, schema)
df.show()

# 1 & 2 → Total spending and number of orders
agg_df = df.groupBy("customer_id") \
    .agg(
        F.sum("amount").alias("total_spending"),
        F.count("order_id").alias("num_orders")
    )

# 3 → Latest order amount per customer
window_spec = Window.partitionBy("customer_id").orderBy(F.col("order_date").desc())

latest_df = df.withColumn("rn", F.row_number().over(window_spec)) \
              .filter(F.col("rn") == 1) \
              .select("customer_id", F.col("amount").alias("latest_order_amount"))

# Join aggregation + latest order
final_df = agg_df.join(latest_df, on="customer_id", how="inner")

# 4 → Rank customers by total spending (highest first)
rank_window = Window.orderBy(F.col("total_spending").desc())

final_df = final_df.withColumn("spending_rank", F.rank().over(rank_window))

final_df.show()

In [0]:
# explode() is a PySpark function that transforms an array or map column
# into multiple rows, creating one row for each element.

from pyspark.sql import functions as F

df = spark.createDataFrame(
    [(1, ["A", "B", "C"]), (2, ["Null"])],
    ["Col1", "Col2"]
)
df.display()

# Before explode:
# +----+---------+
# |Col1|   Col2  |
# +----+---------+
# |  1 | [A,B,C] |
# |  2 | [Null]  |
# +----+---------+

result = df.select(
    "Col1",
    F.explode("Col2").alias("Col2")
)

# After explode:
# +----+-----+
# |Col1|Col2 |
# +----+-----+
# |  1 |  A  |
# |  1 |  B  |
# |  1 |  C  |
# |  2 | Null|
# +----+-----+

result.show()


In [0]:
from pyspark.sql import functions as F

# Sample DataFrame (if needed)
data = [
    (101, "Alice", "Mon", 8),
    (101, "Alice", "Tue", 6),
    (101, "Alice", "Wed", 7),
    (102, "Bob", "Mon", 5),
    (102, "Bob", "Wed", 9),
    (103, "Charlie", "Tue", 8),
    (103, "Charlie", "Thu", 6)
]

columns = ["emp_id", "emp_name", "day", "hours"]
df = spark.createDataFrame(data, columns)

# Pivot + aggregation
result = (
    df.groupBy("emp_id", "emp_name")
      .pivot("day", ["Mon", "Tue", "Wed", "Thu", "Fri"])
      .agg(F.sum("hours"))
      .fillna(0)
)

# Add total_hours column
result = result.withColumn(
    "total_hours",
    F.col("Mon") + F.col("Tue") + F.col("Wed") + F.col("Thu") + F.col("Fri")
)

result.show()


In [0]:
# ── PASS 1 : Expire changed records ──────────────────────────────
spark.sql("""
    MERGE INTO catalog_name.silver_db.employee_scd2 AS trg
    USING (
        SELECT src.*
        FROM catalog_name.silver_db.employee_stage AS src
        INNER JOIN catalog_name.silver_db.employee_scd2 AS cur
            ON  cur.emp_name  = src.emp_name
            AND cur.is_active = 1
        WHERE
            cur.salary      IS DISTINCT FROM src.salary      OR
            cur.designation IS DISTINCT FROM src.designation OR
            cur.department  IS DISTINCT FROM src.department
    ) AS src
    ON  trg.emp_name  = src.emp_name
    AND trg.is_active = 1

    WHEN MATCHED THEN
        UPDATE SET
            trg.is_active = 0,
            trg.end_date  = CURRENT_TIMESTAMP
""")

# ── PASS 2 : Insert new/fresh versions ───────────────────────────
spark.sql("""
    MERGE INTO catalog_name.silver_db.employee_scd2 AS trg
    USING (
        SELECT src.*
        FROM catalog_name.silver_db.employee_stage AS src
        LEFT JOIN catalog_name.silver_db.employee_scd2 AS cur
            ON  cur.emp_name  = src.emp_name
            AND cur.is_active = 1
        WHERE cur.emp_name IS NULL
    ) AS src
    ON  trg.emp_name  = src.emp_name
    AND trg.is_active = 1

    WHEN NOT MATCHED BY TARGET THEN
        INSERT (
            emp_name, hr_name, manager_name, salary,
            location, designation, department,
            doj, dob, rank,
            is_active, start_date, end_date
        )
        VALUES (
            src.emp_name, src.hr_name, src.manager_name, src.salary,
            src.location, src.designation, src.department,
            src.doj, src.dob, src.rank,
            1, CURRENT_TIMESTAMP, NULL
        )
""")

In [0]:
# Find all employees who earn more than the average salary of their own department, and display their name, salary, department name, and how much they earn above the department average. Order the result by the excess amount in descending order.
df = spark.sql("select * from workspace.tirtho_db.employee")
df1 = spark.sql("select * from workspace.tirtho_db.department")

joined_df = df.join(df1, on="DeptID", how="inner")

window_spec = Window.partitionBy("DeptName")

df_avg = joined_df.withColumn("avg_salary", (avg("Salary")).over(window_spec))

result_df = df_avg.filter(col("Salary") > col("avg_salary")).withColumn("sal_diff", round(col("Salary") - col("avg_salary"), 2)).select("EmpName", "DeptName", "Salary", "avg_salary", "sal_diff") \
.orderBy(col("sal_diff").desc())

result_df.display()

In [0]:
df = spark.sql("select * from workspace.tirtho_db.employee")
df1 = spark.sql("select * from workspace.tirtho_db.department")

joined_df = df.join(df1, on="deptid", how="inner")

window_spec = Window.partitionBy("deptname").orderBy(desc("salary"))

df_result = joined_df.withColumn("rank",dense_rank().over(window_spec)).filter(col("rank") <=2)

df_result.display()

In [0]:
##Find employees who joined the company in the same year as at least one other employee from a different department. Display the employee name, department name, joining year, and the count of departments that have at least one joiner in that same year. Order by joining year.

from pyspark.sql import Window
from pyspark.sql.functions import year, col, countDistinct

# Step 1: Load tables
emp_df  = spark.sql("SELECT * FROM workspace.tirtho_db.employee")
dept_df = spark.sql("SELECT * FROM workspace.tirtho_db.department")

# Step 2: Add JoiningYear column
emp_df = emp_df.withColumn("JoiningYear", year(col("JoiningDate")))

# Step 3: Count distinct depts per joining year
dept_count_df = emp_df.groupBy("JoiningYear") \
                      .agg(countDistinct("DeptID").alias("DeptCount")) \
                      .filter(col("DeptCount") > 1)

# Step 4: Join back to employee + department
result_df = emp_df.join(dept_df, on="DeptID", how="inner") \
                  .join(dept_count_df, on="JoiningYear", how="inner") \
                  .select("EmpName", "DeptName", "JoiningYear", "DeptCount") \
                  .orderBy(desc("JoiningYear"))
result_df.display()


In [0]:
from pyspark.sql import Window
from pyspark.sql.functions import year, col, countDistinct, avg

# Step 1: Load tables
emp_df  = spark.sql("SELECT * FROM workspace.tirtho_db.employee")
dept_df = spark.sql("SELECT * FROM workspace.tirtho_db.department")

df_join = emp_df.join(dept_df, on="DeptID", how="inner")

# Calculate overall average salary
overall_avg_salary = df_join.select(avg("salary")).collect()[0][0]

# Calculate average salary per department
df_result = df_join.groupBy("deptname", "deptid").agg(avg("salary").alias("Avg_Salary")).filter(col("Avg_Salary") > overall_avg_salary)
df_result.display()

In [0]:
from pyspark.sql.functions import col, count

# Load tables
emp_df = spark.sql("SELECT * FROM workspace.tirtho_db.employee")
dept_df = spark.sql("SELECT * FROM workspace.tirtho_db.department")

# Self-join: employees with their managers
emp_alias = emp_df.alias("e")
mgr_alias = emp_df.alias("m")

df_join = emp_alias.join(mgr_alias, col("e.ManagerID") == col("m.EmpID"), how="inner")

# ✅ Fix: use dept_df["DeptID"] to avoid ambiguity with m.DeptID
df_join = df_join.join(dept_df, col("m.DeptID") == dept_df["DeptID"], how="inner")

# Group by manager name and department, count employees
df_result = df_join.groupBy(
    col("m.EmpName").alias("ManagerName"),
    col("DeptName")
).agg(count(col("e.EmpID")).alias("Emp_Count")).filter(col("Emp_Count") >= 2).orderBy(col("Emp_Count").desc())

df_result.display()

In [0]:
events = spark.sql("select * from workspace.tirtho_db.user_activity")

# CTE 1: First login per user
first_login = (
    events
    .filter(F.col("event_type") == "login")
    .groupBy("user_id")
    .agg(F.min("event_timestamp").alias("first_login_time"))

)

# CTE 2: First purchase per user
first_purchase = (
    events
    .filter(F.col("event_type") == "purchase")
    .groupBy("user_id")
    .agg(F.min("event_timestamp").alias("first_purchase_time"))
)

# Join + compute hours_to_convert
result = (
    first_login
    .join(first_purchase, on="user_id", how="inner")
    .filter(F.col("first_purchase_time") > F.col("first_login_time"))
    .withColumn(
        "hours_to_convert",
        F.round(
            (F.col("first_purchase_time").cast("long") - F.col("first_login_time").cast("long")) / 3600.0,
            2
        )
    )
    .select("user_id", "first_login_time", "first_purchase_time", "hours_to_convert")
    .orderBy("hours_to_convert")
)

result.show(truncate=False)